### **Student Information**
GitHub ID: singasean

Kaggle name: Seantwx

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

## 1. Model Development (10 pts Required)

**Describe briefly each section, you can add graphs/charts to support your explanations.**

### 1.1 Preprocessing Steps

* **Loading:** read 'final_posts.json' into a DataFrame and join it with 'data_identication.csv' and 'emotion.csv' on 'post_id': 3360 training and 1440 test posts.
*  **Text cleaning:** curly quotes were turned into straight ones, *don’t* and *don't* tokenize the same and repeated spaces and newlines were collapsed. Case, punctuation and emojis were kept because they carry emotion and RoBERTa's tokenizer is case-sensitive.
*   **Duplicates:** 50 training texts appear more than once. Exact repeats were dropped, leaving 3326 training posts. Repeated texts with *different* labels were kept, since a post can genuinely carry 2 emotions.

### 1.2 Feature Engineering Steps

* **TF-IDF first, as a baseline:** word 1-2 grams plus character 2-5 grams with a linear SVM reached only **0.513 macro F1** in 5-fold cross-validation, barely above the 0.5 competition baseline. Posts are short, so word counts give little evidence.
*  **Learned features instead:** posts are converted to subword tokens with RoBERTa's own tokenizer. The model learns its features from these tokens during fine-tuning, rather than from hand-made columns.
*   **Validation split:** 10% of the training posts were held out, stratified by emotion. to choose the best epoch without touching the test set.
*   **Class weights:** the 6 emotions are imbalanced, and macro F1 counts each emotion equally. The loss therefore weights each class by inverse frequency: fear 1.96, disgust 1.23, surprise 1.27, sadness 0.9, anger 0..83, joy 0.64.
### 1.3 Explanation of Your Model

* **Model:** `microsoft/deberta-v3-base`, a pre-trained transformer, with a new 6-class classification layer on top. I fine-tuned it on the training posts and loaded it in float32.
* **Training (each fold):** AdamW (learning rate 2e-5, weight decay 0.01), batch size 16, 6 epochs, 10% warm-up then linear learning rate decay, gradient clipping at 1.0, and the class-weighted loss from 1.2. I trained on a Colab T4 GPU.
* **Model selection:** Each fold keeps the weights from its best epoch. With 4 epochs the out-of-fold macro F1 was 0.579, and several folds were still improving at epoch 4, so I increased it to 6 epochs, which raised the **out-of-fold macro F1 to 0.596**.
* **Ensemble:** All 5 fold models predict the probabilities for the 1440 test posts. I averaged them and picked the emotion with the highest probability.
* **Result:** private leaderboard **0.6056**, public leaderboard **0.6065**. My earlier versions got 0.5688 (`roberta-base`) and 0.6022 (DeBERTa with 4 epochs) on the public leaderboard, and the TF-IDF baseline got 0.513 in cross-validation.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

* I used Claude for this competition. My main prompts are:
* 1. I described the competition task and its goals to the AI, and asked how to approach it. The AI gave me a few options: TF-IDF models and fined tuned-transformers such as RoBERTa, DeBERTa-v3 and RoBERTa-large.
  2. I chose 'roberta-base' and ran the code on Colab as my laptop does not have a dedicated GPU.
  3. After running the code, I asked it if my submission file is in the right format before i submitted it to kaggle.

* **What I decided:**
* **Used DeBERTa-v3 with a 5-fold ensemble and 6 epochs** as my final model, because each change improved my score.
* **Kept light cleaning**, because punctuation and emphasis like "SHOCKED!" show emotion.
* **Used class weights and best-epoch selection**, because macro F1 rewards the smaller classes and some folds got worse in later epochs.

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [ ]:
import re
import numpy as np
import pandas as pd

# Load the three competition files and join them on post_id.
posts = pd.read_json("final_posts.json")
split = pd.read_csv("data_identification.csv")
labels = pd.read_csv("emotion.csv")
df = posts.merge(split, on="post_id").merge(labels, on="post_id", how="left")

def clean(text):
    # Curly quotes -> straight, so "don’t" and "don't" become the same tokens.
    text = text.replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
    # Collapse newlines and repeated spaces. Case, punctuation and emojis stay:
    # "SHOCKED!" carries emotion that lowercasing would remove.
    return re.sub(r"\s+", " ", text).strip()

df["text"] = df["text"].map(clean)

# Drop exact repeats (same text AND same label). Same text with different
# labels is kept: those are posts that genuinely carry two emotions.
train = df[df["set"] == "train"].drop_duplicates(["text", "label"]).reset_index(drop=True)
test = df[df["set"] == "test"].reset_index(drop=True)
print("train:", train.shape, "| test:", test.shape)
print(train["label"].value_counts())

## 2. Feature Engineering Steps

In [ ]:
import torch
from transformers import AutoTokenizer
from sklearn.model_selection import StratifiedKFold

MODEL = "microsoft/deberta-v3-base"
LABELS = sorted(train["label"].unique())
label2id = {label: i for i, label in enumerate(LABELS)}

# DeBERTa-v3's own subword tokenizer: the features are learned by the model,
# not hand-made like TF-IDF columns.
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def make_loader(frame, shuffle, with_labels=True, batch_size=16):
    # Text -> token ids + attention mask; 128 tokens covers about 99% of posts.
    enc = tokenizer(list(frame["text"]), truncation=True, max_length=128,
                    padding=True, return_tensors="pt")
    tensors = [enc["input_ids"], enc["attention_mask"]]
    if with_labels:
        tensors.append(torch.tensor(frame["label"].map(label2id).values))
    dataset = torch.utils.data.TensorDataset(*tensors)
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

# 5-fold cross-validation: each training post is used for training in 4 runs and
# for validation in 1, so no data is permanently held out, and the 5 validation
# folds together score every training post.
N_FOLDS = 5
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=42).split(train, train["label"]))
test_loader = make_loader(test, shuffle=False, with_labels=False)

# Class weights: fear has far fewer posts than joy, and macro F1 counts every
# emotion equally, so mistakes on rare classes cost more in the loss.
counts = train["label"].value_counts().reindex(LABELS).values
class_weights = torch.tensor(len(train) / (len(LABELS) * counts), dtype=torch.float)
print(dict(zip(LABELS, class_weights.numpy().round(2))))

## 3. Model Implementation Steps

In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
EPOCHS = 6

def predict_proba(model, loader):
    # Probability of each emotion for every post in the loader.
    model.eval()
    out = []
    with torch.no_grad():
        for batch in loader:
            ids, mask = batch[0].to(device), batch[1].to(device)
            out.append(torch.softmax(model(input_ids=ids, attention_mask=mask).logits, -1).cpu())
    return torch.cat(out).numpy()

y_all = train["label"].map(label2id).values
oof = np.zeros((len(train), len(LABELS)))          # validation predictions for every training post
test_proba = np.zeros((len(test), len(LABELS)))    # averaged over the 5 fold models

for fold, (tr_idx, val_idx) in enumerate(folds):
    torch.manual_seed(42 + fold)
    tr_loader = make_loader(train.iloc[tr_idx], shuffle=True)
    val_loader = make_loader(train.iloc[val_idx], shuffle=False)

    # A fresh pre-trained DeBERTa-v3 with a new 6-way classification layer per fold.
    model = AutoModelForSequenceClassification.from_pretrained(
      MODEL, num_labels=len(LABELS), dtype=torch.float32).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    total_steps = EPOCHS * len(tr_loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))

    best_f1, best_state, best_val = -1.0, None, None
    for epoch in range(EPOCHS):
        model.train()
        for ids, mask, y in tr_loader:
            ids, mask, y = ids.to(device), mask.to(device), y.to(device)
            loss = loss_fn(model(input_ids=ids, attention_mask=mask).logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        val_proba = predict_proba(model, val_loader)
        f1 = f1_score(y_all[val_idx], val_proba.argmax(1), average="macro")
        print(f"fold {fold + 1} epoch {epoch + 1}: validation macro F1 {f1:.3f}")
        if f1 > best_f1:  # keep the weights of this fold's best epoch
            best_f1, best_val = f1, val_proba
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    oof[val_idx] = best_val
    test_proba += predict_proba(model, test_loader) / N_FOLDS   # ensemble: average the 5 models
    del model, optimizer, best_state
    if device == "cuda":
        torch.cuda.empty_cache()

print("out-of-fold macro F1 (all training posts):",
      round(f1_score(y_all, oof.argmax(1), average="macro"), 3))

# Write the submission from the averaged probabilities.
submission = pd.DataFrame({"post_id": test["post_id"],
                           "label": [LABELS[i] for i in test_proba.argmax(1)]})
submission.to_csv("submission.csv", index=False)
print(submission["label"].value_counts())
from google.colab import files
files.download("submission.csv")   # save the file straight away, before the runtime can reset